# 04 – Final model and test-set evaluation

Finalising the model chosen in `03_models_and_evaluation.ipynb`: choosing the decision threshold, checking one improvement and a time-ordered split, and then evaluating once on the held-out test set.

## 1. Setup

In [1]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import fbeta_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

SEED = 42
TEST_SIZE = 0.2
N_FOLDS = 5

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.13.11
pandas: 3.0.6
scikit-learn: 1.9.1


In [2]:
df = pd.read_csv("../data/ai4i2020.csv")

target = "Machine failure"
drop_cols = ["UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF"]

X = df.drop(columns=drop_cols + [target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

numeric_cols = ["Air temperature [K]", "Process temperature [K]",
                "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
categorical_cols = ["Type"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Settings selected by the grid search in notebook 03, Section 4
final_rf = RandomForestClassifier(
    n_estimators=500, max_depth=None, min_samples_leaf=5, max_features=0.5,
    class_weight="balanced_subsample", random_state=SEED, n_jobs=-1,
)
final_pipe = Pipeline([("prep", preprocessor), ("model", final_rf)])

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (8000, 6) Test: (2000, 6)


## 2. Decision threshold

The model outputs a failure probability, and the threshold decides when that counts as a failure. I compare thresholds using out-of-fold probabilities on the training set only, and pick the one with the highest F2. The test set is not used.

In [3]:
oof_prob = cross_val_predict(final_pipe, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)
rows = []
for t in thresholds:
    pred = (oof_prob >= t).astype(int)
    rows.append({
        "threshold": t,
        "f2": fbeta_score(y_train, pred, beta=2, zero_division=0),
        "precision": precision_score(y_train, pred, zero_division=0),
        "recall": recall_score(y_train, pred, zero_division=0),
        "flagged": int(pred.sum()),
    })

threshold_table = pd.DataFrame(rows).set_index("threshold").round(3)
best_threshold = threshold_table["f2"].idxmax()

print("Best threshold by out-of-fold F2:", best_threshold)
threshold_table

Best threshold by out-of-fold F2: 0.5


,f2,precision,recall,flagged
threshold,,,,
0.10,0.612,0.251,0.956,1033
0.15,0.648,0.293,0.930,860
0.20,0.675,0.334,0.908,737
0.25,0.697,0.371,0.893,652
0.30,0.715,0.417,0.871,566
0.35,0.731,0.467,0.852,495
0.40,0.737,0.520,0.823,429
0.45,0.740,0.574,0.797,376
0.50,0.748,0.647,0.779,326


**Findings:**
- The best threshold is 0.5 (F2 0.748), so the default turns out to be the right choice. The class weighting in the forest has already shifted its probabilities towards the failure class.
- F2 is fairly flat between 0.40 and 0.55 (0.737 to 0.748), so the exact choice matters little in that range.
- Lowering the threshold catches more failures but raises many more alarms. At 0.30, recall rises to 0.871 but the model flags 566 rows instead of 326, and precision falls to 0.417.
- I will keep 0.5. If missed failures turned out to be much more costly in practice, a lower threshold such as 0.30 would be a reasonable alternative.

## 3. Engineered features

The data audit proposed two features: power (torque × rotational speed, converted to watts) and the difference between process and air temperature. I add them and compare the same Random Forest, with the same settings and folds, with and without them. Each new value only uses its own row, so adding them cannot leak information between rows.

In [4]:
from sklearn.base import clone
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_validate


def add_features(data):
    """Add the two engineered features from the data audit. Each row only uses its own values, so there is no leakage."""
    data = data.copy()
    data["Power [W]"] = data["Torque [Nm]"] * data["Rotational speed [rpm]"] * 2 * np.pi / 60
    data["Temp difference [K]"] = data["Process temperature [K]"] - data["Air temperature [K]"]
    return data


X_train_fe = add_features(X_train)

fe_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols + ["Power [W]", "Temp difference [K]"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])
fe_pipe = Pipeline([("prep", fe_preprocessor), ("model", clone(final_rf))])

scoring = {
    "f2": make_scorer(fbeta_score, beta=2, zero_division=0),
    "average_precision": "average_precision",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
}

rows = []
for name, pipe, data in [("Original features", final_pipe, X_train),
                         ("With engineered features", fe_pipe, X_train_fe)]:
    scores = cross_validate(pipe, data, y_train, cv=cv, scoring=scoring)
    row = {"model": name}
    for metric in scoring:
        row[metric] = f"{scores[f'test_{metric}'].mean():.3f} ± {scores[f'test_{metric}'].std():.3f}"
    row["f2 by fold"] = np.round(scores["test_f2"], 3).tolist()
    rows.append(row)

feature_comparison = pd.DataFrame(rows).set_index("model")
feature_comparison

,f2,average_precision,precision,recall,f2 by fold
model,,,,,
Original features,0.748 ± 0.036,0.757 ± 0.047,0.649 ± 0.039,0.779 ± 0.043,"[0.807, 0.712, 0.719, 0.733, 0.769]"
With engineered features,0.807 ± 0.049,0.872 ± 0.025,0.854 ± 0.039,0.797 ± 0.055,"[0.87, 0.72, 0.821, 0.802, 0.824]"


Because the features change the model's probabilities, I re-check the threshold for the new model in the same way as Section 2.

In [5]:
fe_oof_prob = cross_val_predict(fe_pipe, X_train_fe, y_train, cv=cv, method="predict_proba")[:, 1]

fe_rows = []
for t in thresholds:
    pred = (fe_oof_prob >= t).astype(int)
    fe_rows.append({
        "threshold": t,
        "f2": fbeta_score(y_train, pred, beta=2, zero_division=0),
        "precision": precision_score(y_train, pred, zero_division=0),
        "recall": recall_score(y_train, pred, zero_division=0),
        "flagged": int(pred.sum()),
    })

fe_threshold_table = pd.DataFrame(fe_rows).set_index("threshold")
fe_best_threshold = fe_threshold_table["f2"].idxmax()

print("Best threshold by out-of-fold F2:", fe_best_threshold)
fe_threshold_table.round(4)

Best threshold by out-of-fold F2: 0.4


,f2,precision,recall,flagged
threshold,,,,
0.10,0.6513,0.2891,0.9483,889
0.15,0.6972,0.3506,0.9262,716
0.20,0.7387,0.4234,0.9077,581
0.25,0.7554,0.4799,0.8819,498
0.30,0.7759,0.5519,0.8635,424
0.35,0.8044,0.6480,0.8561,358
0.40,0.8082,0.7305,0.8303,308
0.45,0.8059,0.7829,0.8118,281
0.50,0.8078,0.8538,0.7970,253


**Findings:**
- The engineered features improve every metric: F2 rises from 0.748 to 0.807, average precision from 0.757 to 0.872 and precision from 0.649 to 0.854, while recall also rises slightly (0.779 to 0.797). F2 is higher in all 5 folds, so this is a consistent improvement and not noise.
- The reason is that two failure modes are defined almost directly by these quantities. In this dataset, every power failure has a power below 3,500 W or above 9,000 W, and no other row falls outside that range. Every heat dissipation failure has a temperature difference below 8.6 K and a speed below 1,380 rpm. The original features contain this information, but the forest has to approximate the products and differences with many splits.
- This is a limitation as well as a gain. The dataset is synthetic and its failure rules were built from these quantities, so on a real machine the improvement would probably be smaller.
- With the new features, the best threshold is 0.40 (F2 0.8082), but 0.50 is almost identical (0.8078). I follow my rule of choosing the highest out-of-fold F2, so I use 0.40. It catches more failures (recall 0.830 compared with 0.797), at the cost of more alarms (308 compared with 253 flagged).
- I did not re-tune the forest's settings for the new features. They were tuned on the original features, so the new model may be slightly under-tuned, but a second grid search would add little.
- **Decision:** the final model is the Random Forest with the engineered features and a threshold of 0.40.